# Sudoku Image Solver with CNN Digit Recognition

This notebook trains a CNN using the IDX files in `archive (2)` and solves Sudoku puzzles from images.

Run the cells in order.

In [1]:
import os
import struct
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

print('TensorFlow:', tf.__version__)

DATA_DIR = Path('archive (2)')
MODEL_PATH = Path('digit_cnn_model.keras')


TensorFlow: 2.21.0


In [2]:
def load_idx_images(path: Path) -> np.ndarray:
    with open(path, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        if magic != 2051:
            raise ValueError(f'Invalid image file magic number for {path}: {magic}')
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(num, rows, cols)

def load_idx_labels(path: Path) -> np.ndarray:
    with open(path, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        if magic != 2049:
            raise ValueError(f'Invalid label file magic number for {path}: {magic}')
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

x_train = load_idx_images(DATA_DIR / 'train-images.idx3-ubyte')
y_train = load_idx_labels(DATA_DIR / 'train-labels.idx1-ubyte')
x_test = load_idx_images(DATA_DIR / 't10k-images.idx3-ubyte')
y_test = load_idx_labels(DATA_DIR / 't10k-labels.idx1-ubyte')

print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)


Train: (60000, 28, 28) (60000,)
Test : (10000, 28, 28) (10000,)


In [3]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    x_train,
    y_train_cat,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1,
)

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

model.save(MODEL_PATH)
print('Saved model to', MODEL_PATH)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9229 - loss: 0.2486 - val_accuracy: 0.9808 - val_loss: 0.0618
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9772 - loss: 0.0745 - val_accuracy: 0.9888 - val_loss: 0.0429
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9839 - loss: 0.0531 - val_accuracy: 0.9892 - val_loss: 0.0348
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9875 - loss: 0.0403 - val_accuracy: 0.9897 - val_loss: 0.0329
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9889 - loss: 0.0356 - val_accuracy: 0.9905 - val_loss: 0.0351
Test accuracy: 0.9893
Saved model to digit_cnn_model.keras


In [4]:
def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype='float32')

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped, M

def find_sudoku_contour(gray):
    blur = cv2.GaussianBlur(gray, (7, 7), 0)
    thresh = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    thresh = cv2.bitwise_not(thresh)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            return approx.reshape(4, 2)
    return None


In [5]:
def extract_digit(cell):
    h, w = cell.shape
    margin = int(min(h, w) * 0.15)
    roi = cell[margin:h - margin, margin:w - margin]

    thresh = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 40:
        return None

    x, y, w, h = cv2.boundingRect(c)
    digit = thresh[y:y + h, x:x + w]

    size = max(w, h) + 10
    square = np.zeros((size, size), dtype=np.uint8)
    x_off = (size - w) // 2
    y_off = (size - h) // 2
    square[y_off:y_off + h, x_off:x_off + w] = digit

    digit28 = cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)
    digit28 = digit28.astype('float32') / 255.0
    digit28 = np.expand_dims(digit28, axis=(0, -1))
    return digit28

def recognize_grid_digits(warped_gray, model, confidence_threshold=0.65):
    board = np.zeros((9, 9), dtype=int)
    cell_h = warped_gray.shape[0] // 9
    cell_w = warped_gray.shape[1] // 9

    for r in range(9):
        for c in range(9):
            y0, y1 = r * cell_h, (r + 1) * cell_h
            x0, x1 = c * cell_w, (c + 1) * cell_w
            cell = warped_gray[y0:y1, x0:x1]

            digit_img = extract_digit(cell)
            if digit_img is None:
                board[r, c] = 0
                continue

            probs = model.predict(digit_img, verbose=0)[0]
            pred = int(np.argmax(probs))
            conf = float(np.max(probs))
            board[r, c] = pred if conf >= confidence_threshold else 0

    return board


In [6]:
def find_empty(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                return (r, c)
    return None

def is_valid(board, num, pos):
    r, c = pos
    if any(board[r][i] == num and i != c for i in range(9)):
        return False
    if any(board[i][c] == num and i != r for i in range(9)):
        return False

    box_x = c // 3
    box_y = r // 3
    for i in range(box_y * 3, box_y * 3 + 3):
        for j in range(box_x * 3, box_x * 3 + 3):
            if board[i][j] == num and (i, j) != pos:
                return False
    return True

def solve_sudoku(board):
    empty = find_empty(board)
    if not empty:
        return True

    r, c = empty
    for num in range(1, 10):
        if is_valid(board, num, (r, c)):
            board[r][c] = num
            if solve_sudoku(board):
                return True
            board[r][c] = 0
    return False


In [7]:
def overlay_solution_on_original(original_bgr, puzzle_board, solved_board, contour_pts):
    rect = order_points(contour_pts.astype('float32'))
    dst = np.array([[0, 0], [449, 0], [449, 449], [0, 449]], dtype='float32')
    Minv = cv2.getPerspectiveTransform(dst, rect)

    overlay = np.zeros((450, 450, 3), dtype=np.uint8)
    cell = 450 // 9

    for r in range(9):
        for c in range(9):
            if puzzle_board[r, c] == 0:
                text = str(int(solved_board[r, c]))
                x = c * cell + cell // 3
                y = (r + 1) * cell - cell // 4
                cv2.putText(overlay, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

    warped_back = cv2.warpPerspective(overlay, Minv, (original_bgr.shape[1], original_bgr.shape[0]))
    mask = np.any(warped_back != 0, axis=2)
    result = original_bgr.copy()
    result[mask] = cv2.addWeighted(original_bgr, 0.5, warped_back, 1.0, 0)[mask]
    return result

def solve_sudoku_from_image(image_path, model):
    original = cv2.imread(str(image_path))
    if original is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')

    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    contour = find_sudoku_contour(gray)
    if contour is None:
        raise RuntimeError('Could not detect Sudoku board contour.')

    warped_gray, _ = four_point_transform(gray, contour)
    warped_gray = cv2.resize(warped_gray, (450, 450))

    puzzle = recognize_grid_digits(warped_gray, model)
    solution = puzzle.copy()

    if not solve_sudoku(solution):
        raise RuntimeError('Sudoku solver could not find a valid solution.')

    solved_image = overlay_solution_on_original(original, puzzle, solution, contour)
    return puzzle, solution, solved_image


In [8]:
# Replace this with your Sudoku image path
IMAGE_PATH = 'puzzle.jpg'

if MODEL_PATH.exists():
    trained_model = keras.models.load_model(MODEL_PATH)
else:
    trained_model = model

puzzle, solution, solved_img = solve_sudoku_from_image(IMAGE_PATH, trained_model)
print('Detected puzzle:\n', puzzle)
print('Solved board:\n', solution)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(solved_img, cv2.COLOR_BGR2RGB))
plt.title('Solved Sudoku Overlay')
plt.axis('off')
plt.show()


RuntimeError: Sudoku solver could not find a valid solution.